In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report

from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier, 
    VotingClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

sns.set(style="whitegrid")


In [2]:
# Load data
df = pd.read_csv("adult_income.csv")

# Clean strings and encode target
for col in df.select_dtypes(include='object'):
    df[col] = df[col].str.strip()
df['income'] = LabelEncoder().fit_transform(df['income'])

# Encode and scale
X = pd.get_dummies(df.drop(columns='income'))
y = df['income']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)


In [5]:
# For scikit-learn >=1.2
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(), 
    n_estimators=50, 
    random_state=42
)

bagging.fit(X_train, y_train)
bagging_preds = bagging.predict(X_test)


In [6]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)


In [7]:
logreg = LogisticRegression(max_iter=1000)
svc = SVC(probability=True)
tree = DecisionTreeClassifier(max_depth=5)

voting = VotingClassifier(
    estimators=[
        ('lr', logreg), 
        ('svc', svc), 
        ('tree', tree)
    ],
    voting='soft'
)
voting.fit(X_train, y_train)
voting_preds = voting.predict(X_test)


In [8]:
stacking = StackingClassifier(
    estimators=[
        ('lr', logreg), 
        ('svc', svc), 
        ('tree', tree)
    ],
    final_estimator=LogisticRegression(),
    passthrough=True
)
stacking.fit(X_train, y_train)
stacking_preds = stacking.predict(X_test)


In [9]:
ensemble_models = {
    "Bagging": bagging_preds,
    "Random Forest": rf_preds,
    "Voting": voting_preds,
    "Stacking": stacking_preds
}

for name, preds in ensemble_models.items():
    print(f"🔍 {name}")
    print("Accuracy:", accuracy_score(y_test, preds))
    print("Classification Report:")
    print(classification_report(y_test, preds))
    print("=" * 50)


🔍 Bagging
Accuracy: 0.8585905112851221
Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.93      0.91      4942
           1       0.74      0.65      0.69      1571

    accuracy                           0.86      6513
   macro avg       0.81      0.79      0.80      6513
weighted avg       0.85      0.86      0.86      6513

🔍 Random Forest
Accuracy: 0.8616612927990174
Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.93      0.91      4942
           1       0.75      0.64      0.69      1571

    accuracy                           0.86      6513
   macro avg       0.82      0.79      0.80      6513
weighted avg       0.86      0.86      0.86      6513

🔍 Voting
Accuracy: 0.8570551205281745
Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.95      0.91      4942
           1       0.78      0.57      0.66   